# Context Length Thresholding Experiment
This notebook contains the complete code for the Needle-in-a-Haystack evaluation of TinyLlama-1.1B-Chat-v1.0.

In [1]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='cuda', torch_dtype=torch.float16)

test_lengths = [1500, 1750, 2000, 2250, 2500]
target_fact = 'The specific parameter limit for the new AlphaX model is 8.5 billion.'
filler_sentence = 'The rapid advancement of machine learning continues to reshape research. '
question = 'What is the parameter limit for the AlphaX model?'
results = []

for target_length in test_lengths:
    fact_tokens = tokenizer.encode(target_fact, add_special_tokens=False)
    filler_tokens = tokenizer.encode(filler_sentence, add_special_tokens=False)
    repeats_needed = (target_length - len(fact_tokens)) // len(filler_tokens)
    half_repeats = repeats_needed // 2
    context_text = (filler_sentence * half_repeats) + ' ' + target_fact + ' ' + (filler_sentence * half_repeats)
    prompt = f'<|system|>\nYou are a helpful assistant. Answer ONLY based on the provided context.\n<|user|>\nContext: {context_text}\n\nQuestion: {question}\n<|assistant|>\n'
    
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    actual_length = inputs.input_ids.shape[1]
    outputs = model.generate(**inputs, max_new_tokens=30, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    success = '8.5' in response
    results.append({'Target_Length': target_length, 'Actual_Tokens': actual_length, 'Success': success, 'Output': response})

df = pd.DataFrame(results)
df.to_csv('context_degradation_results.csv', index=False)
print('Experiment complete.')